# 🚀 High-Speed EVA-02 9-Tile Keyframe Batch Encoder (Colab G4 / GPU)
Encodes 10k–50k video contact sheets into 9-tile MIL embeddings in minutes using Google Colab G4 / High-VRAM GPU.

### Instructions:
1. **Runtime Type**: In Colab menu, choose **Runtime -> Change runtime type -> G4 GPU (or A100 / L4 / T4)**.
2. Place your `colab_batch.tar.gz` archive in your Google Drive (or upload directly in the file browser).
3. Run the cells below.

In [ ]:
# Step 1: Install Dependencies & Check GPU
!nvidia-smi
!pip install -q open-clip-torch timm torchvision pillow tqdm

In [ ]:
# Step 2: Mount Google Drive (if using Drive)
from google.colab import drive
drive.mount('/content/drive')

# Set the path to your uploaded batch tarball
# e.g. /content/drive/MyDrive/colab_batch.tar.gz or /content/colab_batch.tar.gz
TAR_PATH = '/content/drive/MyDrive/colab_batch.tar.gz'

import os, tarfile
if not os.path.exists(TAR_PATH):
    print(f"[!] {TAR_PATH} not found in Drive. Checking local /content/colab_batch.tar.gz...")
    TAR_PATH = '/content/colab_batch.tar.gz'

print(f"[*] Extracting {TAR_PATH} to high-speed local NVMe...")
with tarfile.open(TAR_PATH, 'r:gz') as tar:
    tar.extractall('/content/work')

print("[*] Extraction complete:", os.listdir('/content/work'))

In [ ]:
# Step 3: High-Throughput 9-Tile Batch Encoder on G4 GPU
import json, time, os, sqlite3
import numpy as np
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.io import decode_jpeg
import open_clip

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True

manifest_file = '/content/work/manifest.json'
mosaics_dir = '/content/work/mosaics'

with open(manifest_file, 'r', encoding='utf-8') as f:
    items = json.load(f)

print(f"[*] Loaded manifest with {len(items):,} items to encode on {DEVICE}.")

# Batch size on G4 GPU: 36 videos = 324 tiles per step!
BATCH_SIZE = 36 if torch.cuda.get_device_properties(0).total_memory > 15e9 else 16
print(f"[*] Using Batch Size: {BATCH_SIZE} videos ({BATCH_SIZE * 9} tiles per forward pass)")

class ColabMosaicDataset(Dataset):
    def __init__(self, item_list):
        self.items = item_list
    def __len__(self):
        return len(self.items)
    def __getitem__(self, idx):
        entry = self.items[idx]
        img_path = os.path.join(mosaics_dir, entry['filename'])
        try:
            with open(img_path, 'rb') as f:
                raw_bytes = f.read()
            im = decode_jpeg(torch.frombuffer(raw_bytes, dtype=torch.uint8))
            _, h, w = im.shape
            tw, th = w // 3, h // 3
            crops = [im[:, r*th:(r+1)*th, c*tw:(c+1)*tw].contiguous() for r in range(3) for c in range(3)]
        except Exception:
            crops = [torch.zeros(3, 180, 320, dtype=torch.uint8) for _ in range(9)]
        return crops, idx

def collate_crops(batch):
    all_crops = []
    indices = []
    for crops_list, idx in batch:
        for c in crops_list:
            resized = F.interpolate(c.unsqueeze(0).float(), size=(336, 336), mode='bilinear', align_corners=False).squeeze(0).to(torch.uint8)
            all_crops.append(resized)
        indices.append(idx)
    return torch.stack(all_crops, dim=0), indices

dataset = ColabMosaicDataset(items)
loader = DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=4, collate_fn=collate_crops, pin_memory=True, prefetch_factor=2)

# Load EVA-02 in Native FP16
print("[*] Loading EVA-02-CLIP-L-14-336 in pure FP16 on Tensor Cores...")
model, _, _ = open_clip.create_model_and_transforms('hf-hub:timm/eva02_large_patch14_clip_336.merged2b_s6b_b61k', device=DEVICE)
model = model.half().eval()

IMAGE_MEAN = torch.tensor([0.48145466, 0.4578275, 0.40821073], device=DEVICE, dtype=torch.float16).view(1, 3, 1, 1)
IMAGE_STD = torch.tensor([0.26862954, 0.26130258, 0.27577711], device=DEVICE, dtype=torch.float16).view(1, 3, 1, 1)

all_9tile_embeddings = []
all_1tile_embeddings = []
processed_indices = []

t0 = time.time()
encoded_count = 0

for flat_uint8, batch_indices in loader:
    B = len(batch_indices)
    flat_half = flat_uint8.to(DEVICE, non_blocking=True).half()
    flat_norm = (flat_half / 255.0).sub_(IMAGE_MEAN).div_(IMAGE_STD)

    with torch.inference_mode():
        feats = model.encode_image(flat_norm)
        feats = F.normalize(feats.float(), dim=-1).half()
        tiles_9 = feats.view(B, 9, 768).cpu().numpy()
        tiles_1 = tiles_9[:, 4, :].copy()

    all_9tile_embeddings.append(tiles_9)
    all_1tile_embeddings.append(tiles_1)
    processed_indices.extend(batch_indices)
    encoded_count += B

    if encoded_count % (BATCH_SIZE * 4) == 0 or encoded_count == len(items):
        elapsed = max(0.1, time.time() - t0)
        rate = encoded_count / elapsed
        print(f"[{encoded_count:,} / {len(items):,}] Speed: {rate:.1f} vids/s ({rate*9:.1f} frames/s)")

final_9tiles = np.concatenate(all_9tile_embeddings, axis=0)
final_1tile = np.concatenate(all_1tile_embeddings, axis=0)

print(f"\n[+] Finished encoding! Shape: {final_9tiles.shape}, dtype: {final_9tiles.dtype}")
np.save('/content/encoded_keyframes_9tiles.npy', final_9tiles)
np.save('/content/encoded_globals.npy', final_1tile)

# Export metadata
encoded_meta = [items[i] for i in processed_indices]
with open('/content/encoded_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(encoded_meta, f)

# Copy output to Drive if mounted
if os.path.exists('/content/drive/MyDrive'):
    !cp /content/encoded_keyframes_9tiles.npy /content/drive/MyDrive/encoded_keyframes_9tiles.npy
    !cp /content/encoded_globals.npy /content/drive/MyDrive/encoded_globals.npy
    !cp /content/encoded_manifest.json /content/drive/MyDrive/encoded_manifest.json
    print("[+] Copied result files to your Google Drive root!")